In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, len(files))

In [ ]:
import os

BASE = "/kaggle/input/datasets/aditipathak17/autoimmune-lichen-planus-psoriasis-normal-skin/3 label classifcation"
CLASSES = ["PSORIASIS", "LICHEN PLANUS", "NORMAL SKIN"]

for c in CLASSES:
    files = os.listdir(os.path.join(BASE, c))
    images = [f for f in files if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    print(c, "| all files:", len(files), "| images:", len(images))

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

def show(cls, n=8):
    folder = os.path.join(BASE, cls)
    names = random.sample(os.listdir(folder), n)
    plt.figure(figsize=(20, 3))
    for i, name in enumerate(names):
        plt.subplot(1, n, i + 1)
        plt.imshow(Image.open(os.path.join(folder, name)))
        plt.axis("off")
    plt.suptitle(cls)
    plt.show()

for c in CLASSES:
    show(c)

In [ ]:
import hashlib
import pandas as pd
from PIL import Image

rows = []
bad = []
for c in CLASSES:
    folder = os.path.join(BASE, c)
    for name in os.listdir(folder):
        if not name.lower().endswith((".jpg", ".jpeg", ".png")):
            continue
        path = os.path.join(folder, name)
        try:
            with Image.open(path) as im:
                im.verify()
            with open(path, "rb") as f:
                h = hashlib.md5(f.read()).hexdigest()
            rows.append({"path": path, "label": c, "md5": h})
        except Exception:
            bad.append(path)

df = pd.DataFrame(rows)
print("before:", df.label.value_counts().to_dict(), "| unreadable:", len(bad))

dups = df[df.duplicated("md5", keep=False)]
print("rows involved in exact duplicates:", len(dups))
cross = dups.groupby("md5").label.nunique()
print("identical files labelled as different classes:", (cross > 1).sum())

In [ ]:
df = df.drop_duplicates("md5", keep="first").reset_index(drop=True)
print("after:", df.label.value_counts().to_dict(), "| total:", len(df))


In [ ]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df["label"], random_state=42)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)

train_df = train_df.assign(split="train")
val_df = val_df.assign(split="val")
test_df = test_df.assign(split="test")

all_df = pd.concat([train_df, val_df, test_df]).reset_index(drop=True)
all_df.to_csv("/kaggle/working/split.csv", index=False)

print(pd.crosstab(all_df["split"], all_df["label"], margins=True))